In [1]:
!pip install jugaad-trader nsepy yfinance pandas numpy matplotlib plotly jupyter requests


[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install jugaad-data


[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from jugaad_data.nse import NSELive

In [4]:
from jugaad_data.nse import NSELive
n = NSELive()
data = n.equities_option_chain("NIFTY")
print(data.keys())

dict_keys(['records', 'filtered'])


In [5]:
!pip install nsepy


[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd

In [7]:
data

{'records': {'timestamp': '02-Apr-2026 15:30:00',
  'underlyingValue': 22713.1,
  'data': [{'expiryDates': '07-Apr-2026',
    'CE': {'strikePrice': 0,
     'expiryDate': None,
     'underlying': None,
     'identifier': None,
     'openInterest': 0,
     'changeinOpenInterest': 0,
     'pchangeinOpenInterest': 0,
     'totalTradedVolume': 0,
     'impliedVolatility': 0,
     'lastPrice': 0,
     'change': 0,
     'totalBuyQuantity': 0,
     'totalSellQuantity': 0,
     'buyPrice1': 0,
     'buyQuantity1': 0,
     'sellPrice1': 0,
     'sellQuantity1': 0,
     'underlyingValue': 0,
     'optionType': None,
     'pchange': 0},
    'PE': {'strikePrice': 20100,
     'expiryDate': '07-04-2026',
     'underlying': 'NIFTY',
     'identifier': 'OPTIDXNIFTY07-04-2026PE20100.00',
     'openInterest': 89331,
     'changeinOpenInterest': 89331,
     'pchangeinOpenInterest': 0,
     'totalTradedVolume': 283756,
     'impliedVolatility': 50.58,
     'lastPrice': 3.8,
     'change': 3.75,
     'total

In [8]:
rows = []
for item in data["records"]["data"]:
    for opt_type in ["CE", "PE"]:
        if opt_type in item:
            d = item[opt_type]
            rows.append({
                "Strike": item["strikePrice"],
                "Expiry": item["expiryDates"],   # ← plural, from item not d
                "Type":   opt_type,
                "IV":     d.get("impliedVolatility"),
                "OI":     d.get("openInterest"),
                "Volume": d.get("totalTradedVolume"),
                "LTP":    d.get("lastPrice")
            })

df = pd.DataFrame(rows)
print("Shape:", df.shape)

Shape: (262, 7)


In [9]:
df.head(10)

,Strike,Expiry,Type,IV,OI,Volume,LTP
0,20100,07-Apr-2026,CE,0.00,0,0,0.00
1,20100,07-Apr-2026,PE,50.58,89331,283756,3.80
2,20150,07-Apr-2026,CE,0.00,0,0,0.00
3,20150,07-Apr-2026,PE,50.01,3490,22240,4.00
4,20200,07-Apr-2026,CE,0.00,3,5,2446.15
5,20200,07-Apr-2026,PE,49.26,172187,500087,4.10
6,20250,07-Apr-2026,CE,0.00,0,0,0.00
7,20250,07-Apr-2026,PE,48.43,14791,93846,4.15
8,20300,07-Apr-2026,CE,0.00,0,0,0.00
9,20300,07-Apr-2026,PE,47.75,15996,178898,4.30


In [10]:
import sys
sys.path.append("src")
from fetch_options_chain import fetch_one_day, fetch_all_history

# test just one day first
df_test = fetch_one_day("2024-01-15")
print(df_test)
print(df_test.columns.tolist())

  No NIFTY options found for 2024-01-15
None


AttributeError: 'NoneType' object has no attribute 'columns'

In [11]:
import requests, zipfile, io, pandas as pd

# download the raw file and inspect it directly
dt_str = "2024-01-15"
url = "https://nsearchives.nseindia.com/content/fo/BhavCopy_NSE_FO_0_0_0_20240115_F_0000.csv.zip"
headers = {"User-Agent": "Mozilla/5.0"}

r = requests.get(url, headers=headers, timeout=15)
print("Status:", r.status_code)

z = zipfile.ZipFile(io.BytesIO(r.content))
print("File inside zip:", z.namelist())

df_raw = pd.read_csv(z.open(z.namelist()[0]))
print("\nAll columns:", df_raw.columns.tolist())
print("\nUnique TckrSymb values (first 20):", df_raw["TckrSymb"].unique()[:20])
print("\nUnique FinInstrmTp values:", df_raw["FinInstrmTp"].unique())

Status: 200
File inside zip: ['BhavCopy_NSE_FO_0_0_0_20240115_F_0000.csv']

All columns: ['TradDt', 'BizDt', 'Sgmt', 'Src', 'FinInstrmTp', 'FinInstrmId', 'ISIN', 'TckrSymb', 'SctySrs', 'XpryDt', 'FininstrmActlXpryDt', 'StrkPric', 'OptnTp', 'FinInstrmNm', 'OpnPric', 'HghPric', 'LwPric', 'ClsPric', 'LastPric', 'PrvsClsgPric', 'UndrlygPric', 'SttlmPric', 'OpnIntrst', 'ChngInOpnIntrst', 'TtlTradgVol', 'TtlTrfVal', 'TtlNbOfTxsExctd', 'SsnId', 'NewBrdLotQty', 'Rmks', 'Rsvd01', 'Rsvd02', 'Rsvd03', 'Rsvd04']

Unique TckrSymb values (first 20): <StringArray>
['AUROPHARMA',       'GNFC', 'MUTHOOTFIN', 'CANFINHOME',       'BPCL',
   'DIVISLAB',   'TATACOMM',      'BSOFT',   'HINDALCO',    'MPHASIS',
        'UPL',     'AUBANK',       'ATUL',      'TITAN',        'IGL',
     'MARUTI',      'SUNTV',   'HDFCLIFE',   'GRANULES',   'AARTIIND']
Length: 20, dtype: str

Unique FinInstrmTp values: <StringArray>
['STO', 'STF', 'IDO', 'IDF']
Length: 4, dtype: str


In [12]:
# check what symbols exist under IDO (index options)
df_ido = df_raw[df_raw["FinInstrmTp"] == "IDO"]
print("Index options symbols:", df_ido["TckrSymb"].unique())
print("Shape:", df_ido.shape)
print(df_ido.head(3))

Index options symbols: <StringArray>
['FINNIFTY', 'MIDCPNIFTY', 'NIFTY', 'BANKNIFTY']
Length: 4, dtype: str
Shape: (5456, 34)
           TradDt       BizDt Sgmt  Src FinInstrmTp  FinInstrmId  ISIN  \
12110  2024-01-15  2024-01-15   FO  NSE         IDO        41855   NaN   
12111  2024-01-15  2024-01-15   FO  NSE         IDO        45950   NaN   
12112  2024-01-15  2024-01-15   FO  NSE         IDO        42308   NaN   

       TckrSymb  SctySrs      XpryDt  ... TtlTradgVol  TtlTrfVal  \
12110  FINNIFTY      NaN  2024-02-06  ...           0        0.0   
12111  FINNIFTY      NaN  2024-01-16  ...           0        0.0   
12112  FINNIFTY      NaN  2024-02-06  ...           0        0.0   

      TtlNbOfTxsExctd SsnId  NewBrdLotQty  Rmks  Rsvd01  Rsvd02  Rsvd03  \
12110               0    F1            40   NaN     NaN     NaN     NaN   
12111               0    F1            40   NaN     NaN     NaN     NaN   
12112               0    F1            40   NaN     NaN     NaN     NaN   

   

In [14]:
script = '''
import os
import time
import requests
import zipfile
import io
import pandas as pd
from datetime import datetime, timedelta

SAVE_DIR = "data/raw"
os.makedirs(SAVE_DIR, exist_ok=True)

def fetch_one_day(date_str):
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    yyyy = dt.strftime("%Y")
    mm   = dt.strftime("%m")
    dd   = dt.strftime("%d")
    url = (
        f"https://nsearchives.nseindia.com/content/fo/"
        f"BhavCopy_NSE_FO_0_0_0_{yyyy}{mm}{dd}_F_0000.csv.zip"
    )
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        r = requests.get(url, headers=headers, timeout=15)
        if r.status_code != 200:
            print(f"  No data for {date_str} (status {r.status_code})")
            return None

        z = zipfile.ZipFile(io.BytesIO(r.content))
        fname = z.namelist()[0]
        df = pd.read_csv(z.open(fname))

        # FIXED FILTERS — IDO = Index Options, NIFTY = Nifty 50
        df = df[df["FinInstrmTp"] == "IDO"]
        df = df[df["TckrSymb"] == "NIFTY"]

        if df.empty:
            print(f"  No NIFTY options for {date_str}")
            return None

        df = df.rename(columns={
            "StrkPric":    "Strike",
            "XpryDt":      "Expiry",
            "OptnTp":      "OptionType",
            "OpnIntrst":   "OI",
            "TtlTradgVol": "Volume",
            "SttlmPric":   "SettlementPrice",
            "TradDt":      "Date"
        })

        keep = ["Date", "Strike", "Expiry", "OptionType", "OI", "Volume", "SettlementPrice"]
        existing = [c for c in keep if c in df.columns]
        df = df[existing]

        # clean up date column
        df["Date"] = date_str

        return df

    except Exception as e:
        print(f"  Error on {date_str}: {e}")
        return None


def get_trading_days(start_str, end_str):
    start = datetime.strptime(start_str, "%Y-%m-%d")
    end   = datetime.strptime(end_str,   "%Y-%m-%d")
    days = []
    current = start
    while current <= end:
        if current.weekday() < 5:
            days.append(current.strftime("%Y-%m-%d"))
        current += timedelta(days=1)
    return days


def fetch_all_history(start_str, end_str):
    trading_days = get_trading_days(start_str, end_str)
    total = len(trading_days)
    print(f"Found {total} weekdays to process")
    success = 0
    skipped = 0
    failed  = 0
    for i, date_str in enumerate(trading_days):
        save_path = os.path.join(SAVE_DIR, f"{date_str}.csv")
        if os.path.exists(save_path):
            skipped += 1
            continue
        print(f"[{i+1}/{total}] Fetching {date_str}...", end=" ")
        df = fetch_one_day(date_str)
        if df is not None:
            df.to_csv(save_path, index=False)
            print(f"saved ({len(df)} rows)")
            success += 1
        else:
            failed += 1
        time.sleep(1.5)
    print(f"\\nDone. Success:{success}  Skipped:{skipped}  Failed:{failed}")
'''

with open("src/fetch_options_chain.py", "w") as f:
    f.write(script)

print("Script updated successfully.")

Script updated successfully.


In [15]:
import importlib, sys

if "fetch_options_chain" in sys.modules:
    del sys.modules["fetch_options_chain"]

sys.path.insert(0, "src")
from fetch_options_chain import fetch_one_day

df_test = fetch_one_day("2024-01-15")
print(df_test.head(10))
print("\nShape:", df_test.shape)
print("Columns:", df_test.columns.tolist())
print("Option types:", df_test["OptionType"].unique())
print("Strikes:", df_test["Strike"].nunique(), "unique strikes")
print("Expiries:", df_test["Expiry"].unique())

             Date   Strike      Expiry OptionType       OI  Volume  \
28868  2024-01-15  21200.0  2024-02-29         CE    36050     341   
28869  2024-01-15  21200.0  2024-02-08         CE        0       0   
28870  2024-01-15  21200.0  2024-02-08         PE     3350      84   
28871  2024-01-15  21200.0  2024-01-18         CE    91600     408   
28872  2024-01-15  21200.0  2024-02-15         PE        0       0   
28873  2024-01-15  21200.0  2024-02-15         CE        0       0   
28874  2024-01-15  21200.0  2024-03-28         CE     2450      30   
28875  2024-01-15  21200.0  2024-02-29         PE   271800    6294   
28876  2024-01-15  21200.0  2024-01-18         PE  3910600  431921   
28877  2024-01-15  21950.0  2024-02-15         CE        0       0   

       SettlementPrice  
28868          1165.60  
28869          1012.80  
28870            62.10  
28871           913.80  
28872            33.15  
28873          1051.80  
28874          1278.50  
28875            95.00  
2887

In [16]:
from fetch_options_chain import fetch_all_history

# test on 5 days first before running full year
fetch_all_history("2024-01-15", "2024-01-19")

Found 5 weekdays to process
[1/5] Fetching 2024-01-15... saved (1540 rows)
[2/5] Fetching 2024-01-16... saved (1596 rows)
[3/5] Fetching 2024-01-17... saved (1596 rows)
[4/5] Fetching 2024-01-18... saved (1600 rows)
[5/5] Fetching 2024-01-19... saved (1595 rows)

Done. Success:5  Skipped:0  Failed:0


In [17]:
import os
files = os.listdir("data/raw")
print(f"Files in data/raw: {len(files)}")
print(files)

Files in data/raw: 5
['2024-01-15.csv', '2024-01-16.csv', '2024-01-17.csv', '2024-01-18.csv', '2024-01-19.csv']


In [18]:
fetch_all_history("2023-06-01", "2024-06-01")

Found 262 weekdays to process
[1/262] Fetching 2023-06-01...   No data for 2023-06-01 (status 404)
[2/262] Fetching 2023-06-02...   No data for 2023-06-02 (status 404)
[3/262] Fetching 2023-06-05...   No data for 2023-06-05 (status 404)
[4/262] Fetching 2023-06-06...   No data for 2023-06-06 (status 404)
[5/262] Fetching 2023-06-07...   No data for 2023-06-07 (status 404)
[6/262] Fetching 2023-06-08...   No data for 2023-06-08 (status 404)
[7/262] Fetching 2023-06-09...   No data for 2023-06-09 (status 404)
[8/262] Fetching 2023-06-12...   No data for 2023-06-12 (status 404)
[9/262] Fetching 2023-06-13...   No data for 2023-06-13 (status 404)
[10/262] Fetching 2023-06-14...   No data for 2023-06-14 (status 404)
[11/262] Fetching 2023-06-15...   No data for 2023-06-15 (status 404)
[12/262] Fetching 2023-06-16...   No data for 2023-06-16 (status 404)
[13/262] Fetching 2023-06-19...   No data for 2023-06-19 (status 404)
[14/262] Fetching 2023-06-20...   No data for 2023-06-20 (status 404)

In [19]:
import os
import pandas as pd

files = sorted([f for f in os.listdir("data/raw") if f.endswith(".csv")])

print(f"Total files saved: {len(files)}")
print(f"First file: {files[0]}")
print(f"Last file:  {files[-1]}")

# check total rows across all files
total_rows = 0
for f in files:
    df = pd.read_csv(f"data/raw/{f}")
    total_rows += len(df)

print(f"Total rows across all files: {total_rows:,}")
print(f"Average rows per day: {total_rows // len(files):,}")

Total files saved: 101
First file: 2024-01-01.csv
Last file:  2024-05-31.csv
Total rows across all files: 153,249
Average rows per day: 1,517


In [20]:
from fetch_options_chain import fetch_all_history
fetch_all_history("2023-11-01", "2023-12-31")

Found 43 weekdays to process
[1/43] Fetching 2023-11-01...   No data for 2023-11-01 (status 404)
[2/43] Fetching 2023-11-02...   No data for 2023-11-02 (status 404)
[3/43] Fetching 2023-11-03...   No data for 2023-11-03 (status 404)
[4/43] Fetching 2023-11-06...   No data for 2023-11-06 (status 404)
[5/43] Fetching 2023-11-07...   No data for 2023-11-07 (status 404)
[6/43] Fetching 2023-11-08...   No data for 2023-11-08 (status 404)
[7/43] Fetching 2023-11-09...   No data for 2023-11-09 (status 404)
[8/43] Fetching 2023-11-10...   No data for 2023-11-10 (status 404)
[9/43] Fetching 2023-11-13...   No data for 2023-11-13 (status 404)
[10/43] Fetching 2023-11-14...   No data for 2023-11-14 (status 404)
[11/43] Fetching 2023-11-15...   No data for 2023-11-15 (status 404)
[12/43] Fetching 2023-11-16...   No data for 2023-11-16 (status 404)
[13/43] Fetching 2023-11-17...   No data for 2023-11-17 (status 404)
[14/43] Fetching 2023-11-20...   No data for 2023-11-20 (status 404)
[15/43] Fetchi

In [21]:
import pandas as pd
import os

dfs = []
for f in sorted(os.listdir("data/raw")):
    if f.endswith(".csv"):
        df = pd.read_csv(f"data/raw/{f}")
        dfs.append(df)

master = pd.concat(dfs, ignore_index=True)
master.to_csv("data/processed/master_raw.csv", index=False)

print("Master dataframe saved.")
print("Shape:", master.shape)
print("Date range:", master["Date"].min(), "to", master["Date"].max())
print("Unique dates:", master["Date"].nunique())
print("Option types:", master["OptionType"].unique())

Master dataframe saved.
Shape: (153249, 7)
Date range: 2024-01-01 to 2024-05-31
Unique dates: 101
Option types: <StringArray>
['CE', 'PE']
Length: 2, dtype: str
